# Gender Classification — Exploratory Data Analysis

Explore the **users** dataset used to train the gender classifier (`male`, `female`, `none`).

Model inputs: `name`, `company`, `age`  
Target: `gender`

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate voyage-analytics project root")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.data.ingestion import DataIngestion

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load data

In [2]:
users_df = DataIngestion().load_users_data()
users_df.head()

,code,company,name,gender,age
0,0,4You,Roy Braun,male,20
1,1,Rainbow,Joseph Holsten,male,21
2,2,CloudFy,Wilma Mcinnis,female,22
3,3,FlyingDrops,Paula Daniel,female,23
4,4,4You,Trina Thomas,none,24


**Insight:** Quick sanity check — each row is one traveler with `name`, `company`, `age`, and the target label `gender`. The `code` column is an ID and is **not** used as a model feature at inference time.

## 2. Dataset overview

In [3]:
print(f"Rows: {len(users_df):,}  |  Columns: {users_df.shape[1]}")
print(f"Duplicate rows: {users_df.duplicated().sum()}")
print(f"Unique travelers (name): {users_df['name'].nunique()}")

users_df.info()
users_df.describe(include="all").T

Rows: 300  |  Columns: 5
Duplicate rows: 0
Unique travelers (name): 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   code     300 non-null    int64 
 1   company  300 non-null    object
 2   name     300 non-null    object
 3   gender   300 non-null    object
 4   age      300 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 11.8+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
code,300.0,NaN,NaN,NaN,149.5,86.746758,0.0,74.75,149.5,224.25,299.0
company,300,4,4You,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,300,10,Roy Braun,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,300,3,female,120,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,300.0,NaN,NaN,NaN,41.25,12.841489,20.0,30.0,41.0,52.0,64.0


**Insight:** ~300 traveler records with only **10 unique names** reused across rows — a small, semi-synthetic dataset. The model will learn mostly from **name patterns** rather than broad demographic coverage.

## 3. Missing values & data quality

In [4]:
missing = users_df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(users_df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct})

,missing,pct
code,0,0.0
company,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0


**Insight:** Zero missing values — the dataset is complete. Imputers in the training pipeline are defensive; they are not strictly needed here.

In [ ]:
print("Age range:", users_df["age"].min(), "to", users_df["age"].max())
print("Companies:", sorted(users_df["company"].unique()))
print("Gender labels:", sorted(users_df["gender"].unique()))

**Insight:** Ages span 20–64 across four companies. All three target labels (`male`, `female`, `none`) are present, which is required for multi-class training.

## 4. Target distribution

In [ ]:
gender_counts = users_df["gender"].value_counts()
gender_counts

**Insight:** Class imbalance is **mild** — `female` (~40%) is largest; `male` and `none` are ~30% each. Track **macro-F1** and per-class recall, not accuracy alone.

In [ ]:
# Bar + pie: compare class counts vs. proportions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_counts.plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518", "#54A24B"])
axes[0].set_title("Gender class counts")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

gender_counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", ylabel="")
axes[1].set_title("Gender class share")

plt.tight_layout()
plt.show()

**Insight:** Bar chart = absolute counts; pie chart = proportional share. No class is extremely rare, so stratified splitting (used in training) is appropriate without oversampling.

## 5. Feature analysis

In [ ]:
# Left: overall age spread | Right: age overlap across gender classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

users_df["age"].plot(kind="hist", bins=20, ax=axes[0], color="#4C78A8", edgecolor="white")
axes[0].set_title("Age distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Frequency")

users_df.boxplot(column="age", by="gender", ax=axes[1])
axes[1].set_title("Age by gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Age")
plt.suptitle("")

plt.tight_layout()
plt.show()

**Insight:** Age overlaps heavily across genders — it is a **weak separator** on its own. Expect the model to rely primarily on **name (TF-IDF)**.

In [ ]:
company_gender = pd.crosstab(users_df["company"], users_df["gender"], normalize="index").round(3)
company_gender

**Insight:** Gender proportions are nearly identical across all four companies. `company` alone is unlikely to be a strong predictor.

In [ ]:
company_gender.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="Set2")
plt.title("Gender mix by company")
plt.xlabel("Company")
plt.ylabel("Proportion")
plt.legend(title="Gender", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Insight:** Stacked bars confirm no company skews toward one gender. Feature importance should rank **`name` >> `age` > `company`**.

In [ ]:
name_gender = users_df.groupby("name")["gender"].agg(lambda s: s.nunique())
ambiguous_names = name_gender[name_gender > 1]

print(f"Names mapped to a single gender: {(name_gender == 1).sum()}")
print(f"Names with multiple gender labels: {len(ambiguous_names)}")
if len(ambiguous_names):
    display(users_df[users_df["name"].isin(ambiguous_names.index)].sort_values("name").head(20))

**Insight:** The same name maps to **different gender labels** across rows (synthetic cycling). This label noise caps achievable accuracy and explains the `none` class.

## 6. Modeling notes

- The classifier uses **TF-IDF on `name`**, **one-hot encoding on `company`**, and **scaled `age`** (see `src/models/classification/preprocess.py`).
- Check class balance before choosing metrics; macro-F1 is used during training.
- Ambiguous names (same name, different gender labels) may add label noise.

In [ ]:
summary = {
    "records": len(users_df),
    "gender_classes": users_df["gender"].nunique(),
    "companies": users_df["company"].nunique(),
    "unique_names": users_df["name"].nunique(),
    "age_median": users_df["age"].median(),
    "majority_class": gender_counts.idxmax(),
    "majority_class_pct": round(gender_counts.max() / len(users_df) * 100, 1),
}
pd.Series(summary, name="gender_classification_eda")

**Insight — key takeaways:** Small data + name reuse → overfitting risk. Mild imbalance → use macro-F1. Ambiguous labels → moderate performance is expected.